# 04 — Baseline Models: Logistic Regression + Random Forest

**Input:**
- `notebooks/data/processed/X_baseline_train.parquet`
- `notebooks/data/processed/X_baseline_test.parquet`
- `notebooks/data/processed/y_train.csv`
- `notebooks/data/processed/y_test.csv`
- `notebooks/data/processed/split_metadata.json`

**Models:**
- Logistic Regression (`class_weight='balanced'`)
- Random Forest (`class_weight='balanced'`)
- Majority-class baseline (dummy classifier)

**Output:**
- `notebooks/models/metadata_v1.json` — hyperparams, feature list, metrics (committed)
- `notebooks/reports/eval_v1.md` — narrative evaluation report (committed)
- `notebooks/models/lr_v1.pkl` — LR binary (gitignored)
- `notebooks/models/rf_v1.pkl` — RF binary (gitignored)

## Imports

In [ ]:
import json
import warnings
from datetime import date
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)

PROCESSED_DIR = Path("notebooks/data/processed")
MODELS_DIR    = Path("notebooks/models")
REPORTS_DIR   = Path("notebooks/reports")
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

CV_FOLDS     = 5
RANDOM_STATE = 42

## Load processed data

In [ ]:
X_train = pd.read_parquet(PROCESSED_DIR / "X_baseline_train.parquet")
X_test  = pd.read_parquet(PROCESSED_DIR / "X_baseline_test.parquet")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv").squeeze()
y_test  = pd.read_csv(PROCESSED_DIR / "y_test.csv").squeeze()

with open(PROCESSED_DIR / "split_metadata.json") as f:
    split_meta = json.load(f)

FEATURE_COLS = split_meta["baseline_features"]

print(f"Train : {X_train.shape}  |  at_risk rate : {y_train.mean()*100:.1f}%")
print(f"Test  : {X_test.shape}   |  at_risk rate : {y_test.mean()*100:.1f}%")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

## Define models

In [ ]:
models = {
    # majority-class baseline — no class_weight, always predicts most frequent label
    "majority_baseline": DummyClassifier(
        strategy="most_frequent",
        random_state=RANDOM_STATE,
    ),
    # class_weight="balanced" to handle at_risk class imbalance
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=RANDOM_STATE,
        )),
    ]),
    # class_weight="balanced" to handle at_risk class imbalance
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

print("Models defined:")
for name in models:
    print(f"  {name}")

## Cross-validation (5-fold stratified)

In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_scoring = ["roc_auc", "f1", "precision", "recall"]

cv_results = {}
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=cv_scoring, n_jobs=-1)
    cv_results[name] = {
        "roc_auc_mean":   round(scores["test_roc_auc"].mean(),   4),
        "roc_auc_std":    round(scores["test_roc_auc"].std(),    4),
        "f1_mean":        round(scores["test_f1"].mean(),        4),
        "precision_mean": round(scores["test_precision"].mean(), 4),
        "recall_mean":    round(scores["test_recall"].mean(),    4),
    }
    print(f"{name}: AUC={cv_results[name]['roc_auc_mean']:.4f} ± {cv_results[name]['roc_auc_std']:.4f}  "
          f"F1={cv_results[name]['f1_mean']:.4f}")

cv_df = pd.DataFrame(cv_results).T
display(cv_df)

## Train on full train set and evaluate on held-out test set

In [ ]:
test_results = {}
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    y_pred  = model.predict(X_test)
    y_proba = (
        model.predict_proba(X_test)[:, 1]
        if hasattr(model, "predict_proba")
        else np.zeros(len(y_test))
    )

    test_results[name] = {
        "roc_auc":   round(roc_auc_score(y_test, y_proba) if y_proba.any() else 0.5, 4),
        "f1":        round(f1_score(y_test, y_pred, zero_division=0),        4),
        "precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "recall":    round(recall_score(y_test, y_pred, zero_division=0),    4),
    }

    print(f"\n── {name} ──")
    print(classification_report(y_test, y_pred, target_names=["not_at_risk", "at_risk"]))

test_df = pd.DataFrame(test_results).T
display(test_df)

## Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4))
if len(models) == 1:
    axes = [axes]

for ax, (name, model) in zip(axes, fitted_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["not_at_risk", "at_risk"]).plot(
        ax=ax, colorbar=False
    )
    ax.set_title(name)

plt.tight_layout()
plt.savefig(REPORTS_DIR / "confusion_matrices.png", dpi=150)
plt.show()

## ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

for name, model in fitted_models.items():
    if not hasattr(model, "predict_proba"):
        continue
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)

ax.plot([0, 1], [0, 1], "k--", label="random")
ax.set_title("ROC curves — held-out test set")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "roc_curves.png", dpi=150)
plt.show()

## Feature importance

In [ ]:
# ── Random Forest: Gini importance ───────────────────────────────────────────
rf = fitted_models["random_forest"]
rf_importance = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(6, 5))
rf_importance.plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Random Forest — feature importance (Gini)")
ax.set_xlabel("Mean decrease in impurity")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "rf_feature_importance.png", dpi=150)
plt.show()

# ── Logistic Regression: coefficient magnitudes ───────────────────────────────
lr_clf = fitted_models["logistic_regression"].named_steps["clf"]
lr_coef = pd.Series(lr_clf.coef_[0], index=FEATURE_COLS).sort_values(key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(6, 5))
colors = ["tomato" if v > 0 else "steelblue" for v in lr_coef]
lr_coef.plot(kind="barh", ax=ax, color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Logistic Regression — coefficients (scaled)")
ax.set_xlabel("Coefficient value")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "lr_coefficients.png", dpi=150)
plt.show()

## Beats majority baseline check

In [ ]:
baseline_f1  = test_results["majority_baseline"]["f1"]
baseline_auc = test_results["majority_baseline"]["roc_auc"]

for name in ["logistic_regression", "random_forest"]:
    f1_ok  = test_results[name]["f1"]      > baseline_f1
    auc_ok = test_results[name]["roc_auc"] > baseline_auc
    status = "[PASS]" if (f1_ok and auc_ok) else "[WARN]"
    print(f"{status} {name}: F1={test_results[name]['f1']:.4f} (baseline={baseline_f1:.4f})  "
          f"AUC={test_results[name]['roc_auc']:.4f} (baseline={baseline_auc:.4f})")

## Save model artifacts

In [ ]:
joblib.dump(fitted_models["logistic_regression"], MODELS_DIR / "lr_v1.pkl")
joblib.dump(fitted_models["random_forest"],       MODELS_DIR / "rf_v1.pkl")

metadata = {
    "model_version":    "v1",
    "training_date":    str(date.today()),
    "sklearn_version":  sklearn.__version__,
    "snapshot_date":    split_meta["snapshot_date"],
    "batch_code":       split_meta["batch_code"],
    "split_method":     split_meta["split_method"],
    "group_key":        split_meta["group_key"],
    "baseline_features": FEATURE_COLS,
    "cv_folds":         CV_FOLDS,
    "hyperparameters": {
        "logistic_regression": {
            "class_weight": "balanced",
            "max_iter":     1000,
            "scaler":       "StandardScaler",
        },
        "random_forest": {
            "n_estimators": 200,
            "class_weight": "balanced",
        },
    },
    "cv_metrics":   cv_results,
    "test_metrics": test_results,
}

with open(MODELS_DIR / "metadata_v1.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
print(f"  {MODELS_DIR}/lr_v1.pkl           (gitignored)")
print(f"  {MODELS_DIR}/rf_v1.pkl           (gitignored)")
print(f"  {MODELS_DIR}/metadata_v1.json    (committed)")

## Write eval_v1.md

In [ ]:
report = f"""# Phase 3 — Baseline Model Evaluation (v1)

**Training date:** {date.today()}  
**Snapshot:** {split_meta['snapshot_date']} / batch `{split_meta['batch_code']}`  
**scikit-learn:** {sklearn.__version__}

## Dataset

| | Rows | at_risk rate |
|---|---|---|
| Train | {split_meta['train_rows']} | {split_meta['at_risk_rate_train']*100:.1f}% |
| Test  | {split_meta['test_rows']}  | {split_meta['at_risk_rate_test']*100:.1f}%  |

Split: `GroupShuffleSplit` on `academy_member_id` (no student overlap between train/test)

## Baseline features ({len(FEATURE_COLS)})

{chr(10).join(f'- `{f}`' for f in FEATURE_COLS)}

## Cross-validation results (5-fold stratified, train set)

| Model | ROC-AUC | F1 | Precision | Recall |
|---|---|---|---|---|
""" + \
    "\n".join(
        f"| {name} | {r['roc_auc_mean']:.4f} ± {r['roc_auc_std']:.4f} "
        f"| {r['f1_mean']:.4f} | {r['precision_mean']:.4f} | {r['recall_mean']:.4f} |"
        for name, r in cv_results.items()
    ) + f"""

## Held-out test set results

| Model | ROC-AUC | F1 | Precision | Recall |
|---|---|---|---|---|
""" + \
    "\n".join(
        f"| {name} | {r['roc_auc']:.4f} | {r['f1']:.4f} | {r['precision']:.4f} | {r['recall']:.4f} |"
        for name, r in test_results.items()
    ) + """

## Artifacts

| File | Status |
|---|---|
| `models/lr_v1.pkl` | gitignored — local only |
| `models/rf_v1.pkl` | gitignored — local only |
| `models/metadata_v1.json` | committed |
| `reports/confusion_matrices.png` | local |
| `reports/roc_curves.png` | local |
| `reports/rf_feature_importance.png` | local |
| `reports/lr_coefficients.png` | local |

## Notes

- `majority_baseline`: DummyClassifier, no class_weight — always predicts most frequent label.
- `logistic_regression`: class_weight="balanced" to handle at_risk class imbalance.
- `random_forest`: class_weight="balanced" to handle at_risk class imbalance.
- Oracle/ablation features (2C3L criterion scores) are excluded from this evaluation.
- Teacher-facing prediction API deferred to Phase 4.
"""

with open(REPORTS_DIR / "eval_v1.md", "w", encoding="utf-8") as f:
    f.write(report)

print(f"Saved: {REPORTS_DIR}/eval_v1.md")

## Summary

In [ ]:
print("=" * 55)
print("M3.4 Baseline model evaluation complete")
print("=" * 55)
for name, r in test_results.items():
    print(f"  {name:25s}  AUC={r['roc_auc']:.4f}  F1={r['f1']:.4f}")
print("=" * 55)
print("Committed artifacts:")
print(f"  models/metadata_v1.json")
print(f"  reports/eval_v1.md")
print("=" * 55)
print("Phase 3 notebooks complete. Next: Phase 4 (Risk API).")